# CSE374 – Machine Learning Classification Project
## Credit Card Default Prediction (Taiwan Dataset)

**Dataset:** Default of Credit Card Clients (UCI / Taiwan)  
**Team Members:** Mena Moheb Abdelshaheed · George Ibrahim Abdo · Abdalla Ragaee Ahmed  
**Date:** May 2026  


---
## 1. Problem Understanding


### What problem are we solving?

We are solving a **binary classification problem**: given a credit card client's demographic information and 6 months of payment history, predict whether they will **default on their payment next month** (target = 1) or not (target = 0).

This is a real-world credit risk problem. Banks use such models to flag high-risk clients early — for example, to reduce credit limits, offer payment plans, or increase monitoring.

### Dataset overview
- **Source:** UCI Machine Learning Repository — Default of Credit Card Clients (Taiwan)
- **Size:** 30,000 clients, 23 original features + 1 target
- **Target variable:** `default payment next month` — binary (0 = no default, 1 = default)

### Feature groups
| Group | Features | Description |
|---|---|---|
| Demographics | SEX, EDUCATION, MARRIAGE, AGE | Client profile |
| Credit | LIMIT_BAL | Approved credit limit |
| Payment status | PAY_0 to PAY_6 | Monthly repayment status (-2 to 8) |
| Bill amounts | BILL_AMT1–6 | Monthly bill statements (NT$) |
| Payment amounts | PAY_AMT1–6 | Actual monthly payments made |

### Expected challenges
1. **Class imbalance:** only ~22% of clients default — accuracy alone will be misleading
2. **Skewed distributions:** bill and payment amounts are right-skewed with extreme values
3. **Noisy payment codes:** PAY_X includes undocumented values (0, -2) needing investigation
4. **Correlated features:** bill amounts across months are highly correlated
5. **Borderline cases:** financially stressed clients who do not default look similar to those who do


---
## 2. Data Loading


We load the dataset from the UCI repository. The Excel file has the header on row 1 (row 0 is a description), so we skip it with `header=1`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & splitting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# Classifiers — ONLY allowed models per project rubric
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, classification_report
)

# Imbalance handling
from imblearn.over_sampling import SMOTE

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42

print('All libraries loaded successfully!')


In [ ]:
# Load dataset from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
df_raw = pd.read_excel(url, header=1)

# Rename the target column to something cleaner
df_raw = df_raw.rename(columns={'default payment next month': 'default'})

print(f'Dataset shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
print(f'\nSample (first 3 rows):')
df_raw.head(3)


---
## 3. Data Exploration (EDA)

Before cleaning or modeling, we explore the data to understand its structure, detect issues, and guide preprocessing.


In [ ]:
# Basic statistics
print("=" * 55)
print("BASIC DATASET SUMMARY")
print("=" * 55)
print(f'Shape: {df_raw.shape}')
print(f'Total clients: {len(df_raw):,}')
print(f'\nDescriptive statistics (first 5 columns):')
df_raw.iloc[:, :5].describe().round(2)


### Plot 1: Class Distribution

The first thing to check is how balanced the target variable is — this drives all our subsequent decisions about metrics and imbalance handling.


In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

class_counts = df_raw['default'].value_counts()
class_labels = ['No Default (0)', 'Default (1)']

axes[0].bar(class_labels, class_counts.values, color=['#3498db', '#e74c3c'], alpha=0.85, edgecolor='black')
axes[0].set_title('Class Distribution (Count)', fontsize=13)
axes[0].set_ylabel('Number of Clients')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(class_counts.values, labels=class_labels, autopct='%1.1f%%',
            colors=['#3498db', '#e74c3c'], startangle=90)
axes[1].set_title('Class Distribution (Percentage)', fontsize=13)

plt.suptitle('Target Variable: Default Payment Next Month', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Non-defaulters: {class_counts[0]:,} ({class_counts[0]/len(df_raw)*100:.1f}%)')
print(f'Defaulters:     {class_counts[1]:,} ({class_counts[1]/len(df_raw)*100:.1f}%)')
print(f'Imbalance ratio: {class_counts[0]/class_counts[1]:.1f}:1')


**Interpretation:** About 22% of clients default. A naive model predicting "no default" every time would get 78% accuracy — so accuracy is a misleading metric. We must focus on F1-score and recall for the default class.


### Plot 2: Payment Status (PAY_0) vs Default Rate

PAY_0 is the most recent month's payment status and likely one of the strongest predictors.


In [ ]:
# Default rate by PAY_0 status
pay_default = df_raw.groupby('PAY_0')['default'].agg(['mean', 'count']).reset_index()
pay_default.columns = ['PAY_0', 'default_rate', 'count']
pay_default['default_rate'] = pay_default['default_rate'] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(pay_default['PAY_0'].astype(str), pay_default['default_rate'],
              color='#e74c3c', alpha=0.85, edgecolor='black')
ax.set_xlabel('PAY_0 Status Code', fontsize=12)
ax.set_ylabel('Default Rate (%)', fontsize=12)
ax.set_title('Default Rate by Most Recent Payment Status (PAY_0)', fontsize=13)

for bar, cnt in zip(bars, pay_default['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'n={cnt:,}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print('PAY_0 codes: -2=no consumption, -1=paid in full, 0=revolving credit, 1+=months overdue')


**Interpretation:** Clear monotonic relationship — higher delay codes mean higher default rates. Clients 2+ months late default at 60–70%+. This confirms payment status is the strongest predictor in the dataset.


### Plot 3: Credit Limit by Default Status


In [ ]:
# Credit limit distribution by default group
fig, ax = plt.subplots(figsize=(9, 5))

no_default_lim = df_raw[df_raw['default'] == 0]['LIMIT_BAL']
defaulters_lim  = df_raw[df_raw['default'] == 1]['LIMIT_BAL']

ax.hist(no_default_lim / 1000, bins=40, alpha=0.6, color='#3498db', label='No Default', density=True)
ax.hist(defaulters_lim  / 1000, bins=40, alpha=0.6, color='#e74c3c', label='Default',    density=True)
ax.set_xlabel('Credit Limit (NT$1,000)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Credit Limit Distribution: Defaulters vs Non-Defaulters', fontsize=13)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f'Non-defaulters median credit limit: NT${no_default_lim.median():,.0f}')
print(f'Defaulters     median credit limit: NT${defaulters_lim.median():,.0f}')


**Interpretation:** Defaulters tend to have lower credit limits (lower median), which makes sense — banks grant higher limits to more trustworthy clients. But the distributions overlap heavily, so credit limit alone is not enough for classification.


### Plot 4: Correlation Heatmap


In [ ]:
# Correlation heatmap for key numeric features
key_cols = ['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3',
             'BILL_AMT1', 'BILL_AMT2', 'PAY_AMT1', 'PAY_AMT2', 'default']

corr_matrix = df_raw[key_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Heatmap (Key Features)', fontsize=13)
plt.tight_layout()
plt.show()


**Interpretation:** BILL_AMT columns are highly correlated with each other (~0.9+) — someone with a large balance this month likely had one last month too. PAY_0 shows the highest correlation with the target. This multicollinearity is why we will engineer aggregate features rather than using all raw monthly columns redundantly.


---
## 4. Data Cleaning

We drop the ID column (no predictive value), check for missing values, and fix undocumented category codes.


In [ ]:
# Drop ID column
df = df_raw.drop(columns=['ID'])

# Check for missing values
print("Missing values per column:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("  No missing values found!")
else:
    print(missing[missing > 0])

print(f'\nDataset shape after dropping ID: {df.shape}')


### Fixing Undocumented Category Codes

The documentation says EDUCATION should be 1–4 and MARRIAGE should be 1–3, but the data contains 0, 5, 6 for EDUCATION and 0 for MARRIAGE. These are undocumented. Rather than dropping rows, we merge them into the "others" category since we cannot know what they mean.


In [ ]:
# Check before fixing
print("EDUCATION value counts (before):")
print(df['EDUCATION'].value_counts().sort_index())
print("\nMARRIAGE value counts (before):")
print(df['MARRIAGE'].value_counts().sort_index())

# Merge undocumented EDUCATION codes (0, 5, 6) into 'Others' (4)
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

# Merge undocumented MARRIAGE code (0) into 'Others' (3)
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

print("\nEDUCATION value counts (after):")
print(df['EDUCATION'].value_counts().sort_index())
print("\nMARRIAGE value counts (after):")
print(df['MARRIAGE'].value_counts().sort_index())


---
## 5. Outlier Analysis

### Why outliers matter for this dataset

Financial data like bill amounts are naturally right-skewed with extreme values. One client might have a bill of NT$1,000,000 while most clients have NT$50,000. These extremes can distort models that are sensitive to scale (Logistic Regression, kNN, SVM, Neural Networks).

We use the **IQR method** to count outliers, then apply **winsorization** (capping at the 1st and 99th percentile). This is safer than dropping rows — we keep all 30,000 samples.


In [ ]:
# Define financial columns
bill_cols    = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
pay_amt_cols = ['PAY_AMT1',  'PAY_AMT2',  'PAY_AMT3',  'PAY_AMT4',  'PAY_AMT5',  'PAY_AMT6']
pay_cols     = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

# Count IQR-based outliers
print(f"{'Column':<15} {'Outliers':<12} {'Outlier %'}")
print("-" * 38)

for col in bill_cols + pay_amt_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col:<15} {n_out:<12} {n_out/len(df)*100:.1f}%")


In [ ]:
# Boxplots showing outliers in BILL_AMT columns
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, col in enumerate(bill_cols):
    axes[i].boxplot(df[col] / 1000, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#3498db', alpha=0.6))
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel('Amount (NT$1,000)')

plt.suptitle('Bill Amount Outliers (Before Winsorization)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Applying Winsorization

We cap values at the 1st and 99th percentile. This preserves the ranking of values and reduces outlier influence without removing any rows. We chose 99th percentile because most outlier mass is in the top 1%, and typical financial variation should be preserved.


In [ ]:
# Winsorize all bill and payment columns
financial_cols = bill_cols + pay_amt_cols
df_clean = df.copy()

for col in financial_cols:
    lower_cap = df_clean[col].quantile(0.01)
    upper_cap = df_clean[col].quantile(0.99)
    df_clean[col] = df_clean[col].clip(lower=lower_cap, upper=upper_cap)

print("Winsorization applied to all bill and payment columns.")
print(f"\nBILL_AMT1 before: Max={df['BILL_AMT1'].max():,.0f},  Std={df['BILL_AMT1'].std():,.0f}")
print(f"BILL_AMT1 after:  Max={df_clean['BILL_AMT1'].max():,.0f},  Std={df_clean['BILL_AMT1'].std():,.0f}")
print(f"\nDataset shape unchanged: {df_clean.shape}")


---
## 6. Feature Engineering

Raw features describe individual months in isolation. But default risk is about **patterns over time**. We engineer features that capture these behavioral patterns explicitly. Each feature has an intuitive financial meaning.


### Feature 1: `max_delay` — Worst payment delay across 6 months

**Intuition:** If a client was ever 3+ months late, that is a serious red flag regardless of other months. We take the maximum PAY_X value.

**Why it helps:** Isolates the worst-case signal which may be diluted by an average.


In [ ]:
df_clean['max_delay'] = df_clean[pay_cols].max(axis=1)
print("max_delay distribution:")
print(df_clean['max_delay'].value_counts().sort_index())


### Feature 2: `num_delays` — Count of months with a positive delay

**Intuition:** A client late once differs from one consistently late. This counts months where PAY_X > 0 (at least 1 month overdue).

**Why it helps:** Captures frequency of bad behavior, not just severity.


In [ ]:
df_clean['num_delays'] = (df_clean[pay_cols] > 0).sum(axis=1)
print("num_delays distribution:")
print(df_clean['num_delays'].value_counts().sort_index())


### Feature 3: `avg_pay_status` — Average payment status over 6 months

**Intuition:** Captures the typical repayment behavior over the observation window. Average of -1 means always paying in full; average of 1.5 means consistently behind.


In [ ]:
df_clean['avg_pay_status'] = df_clean[pay_cols].mean(axis=1)
print(f"avg_pay_status — mean: {df_clean['avg_pay_status'].mean():.3f}, std: {df_clean['avg_pay_status'].std():.3f}")


### Features 4 & 5: `total_bill` and `total_paid` — Aggregate financial flow

**Intuition:** Total bill shows overall exposure; total paid shows how much was actually repaid over 6 months. Together they give a full-picture view of the client's financial activity.


In [ ]:
df_clean['total_bill'] = df_clean[bill_cols].sum(axis=1)
df_clean['total_paid'] = df_clean[pay_amt_cols].sum(axis=1)

print(f"total_bill — median: {df_clean['total_bill'].median():,.0f}")
print(f"total_paid — median: {df_clean['total_paid'].median():,.0f}")


### Feature 6: `payment_ratio` — What fraction of total bills was actually paid?

**Intuition:** A client paying 90% of their bills is very different from one paying 10%. This is the most informative engineered feature.

**Why it helps:** Normalizes payment by bill level — a high-limit client might pay NT$100k/month but still be struggling if their bill is NT$500k.


In [ ]:
# Add 1 to avoid division by zero
df_clean['payment_ratio'] = df_clean['total_paid'] / (df_clean['total_bill'].abs() + 1)
# Clip extreme values — ratios above 3 are effectively "fully paid"
df_clean['payment_ratio'] = df_clean['payment_ratio'].clip(upper=3.0)

print(f"payment_ratio — mean: {df_clean['payment_ratio'].mean():.3f}, median: {df_clean['payment_ratio'].median():.3f}")


### Feature 7: `util_rate` — Credit utilization rate

**Intuition:** Using 90% of a credit limit signals financial stress. This is a standard metric in credit risk modeling.


In [ ]:
df_clean['util_rate'] = df_clean['BILL_AMT1'] / (df_clean['LIMIT_BAL'] + 1)
df_clean['util_rate'] = df_clean['util_rate'].clip(lower=-1.0, upper=2.0)

print(f"util_rate — mean: {df_clean['util_rate'].mean():.3f}, std: {df_clean['util_rate'].std():.3f}")


### Feature 8: `avg_bill_change` — Trend in monthly bill amounts

**Intuition:** Growing bills month over month means the client is accumulating debt — a risk signal. A declining trend means they are paying it down.


In [ ]:
# bill_cols is newest-to-oldest (AMT1=newest, AMT6=oldest)
# Reverse to oldest-to-newest before computing differences
bill_values   = df_clean[bill_cols].values
bill_reversed = bill_values[:, ::-1]        # oldest to newest
bill_diffs    = np.diff(bill_reversed, axis=1)  # 5 month-to-month differences

df_clean['avg_bill_change'] = bill_diffs.mean(axis=1)

print(f"avg_bill_change — mean: {df_clean['avg_bill_change'].mean():,.0f}")
print(f"% clients with growing debt (positive change): {(df_clean['avg_bill_change'] > 0).mean()*100:.1f}%")


### Feature 9: `paid_vs_billed` — How much of last month's bill was paid?

**Intuition:** The most recent month is most predictive. A very low ratio (near 0) means the client barely touched their last bill — a strong warning sign.


In [ ]:
df_clean['paid_vs_billed'] = df_clean['PAY_AMT1'] / (df_clean['BILL_AMT1'].abs() + 1)
df_clean['paid_vs_billed'] = df_clean['paid_vs_billed'].clip(upper=3.0)

print(f"paid_vs_billed — mean: {df_clean['paid_vs_billed'].mean():.3f}")


### Feature 10: `revolving` — Did the client ever fully pay their bill?

**Intuition:** A revolving user never fully pays their balance — always carrying forward debt and paying interest. These clients are generally higher risk.


In [ ]:
# revolving = 1 if the client NEVER fully paid any bill in 6 months
is_revolving = np.ones(len(df_clean), dtype=int)

for i in range(1, 7):
    pa_col = f'PAY_AMT{i}'
    ba_col = f'BILL_AMT{i}'
    # If payment ever >= bill (for a positive bill), client is NOT purely revolving
    paid_enough = (df_clean[pa_col] >= df_clean[ba_col]) & (df_clean[ba_col] > 0)
    is_revolving = is_revolving & (~paid_enough).astype(int)

df_clean['revolving'] = is_revolving

print(f"Revolving users: {df_clean['revolving'].sum():,} ({df_clean['revolving'].mean()*100:.1f}%)")
print(f"Default rate — revolving:     {df_clean[df_clean['revolving']==1]['default'].mean()*100:.1f}%")
print(f"Default rate — non-revolving: {df_clean[df_clean['revolving']==0]['default'].mean()*100:.1f}%")


### Verifying engineered features correlate with target


In [ ]:
new_features = ['max_delay', 'num_delays', 'avg_pay_status', 'total_bill',
                'total_paid', 'payment_ratio', 'util_rate', 'avg_bill_change',
                'paid_vs_billed', 'revolving']

feat_corr = df_clean[new_features + ['default']].corr()['default'].drop('default').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if c > 0 else '#3498db' for c in feat_corr.values]
ax.barh(feat_corr.index, feat_corr.values, color=colors, alpha=0.85, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with Default', fontsize=12)
ax.set_title('Engineered Feature Correlations with Target Variable', fontsize=13)
plt.tight_layout()
plt.show()

print(feat_corr.round(3))


**Interpretation:** `max_delay` and `num_delays` are the most strongly correlated with default — as expected. `payment_ratio` and `avg_pay_status` also have meaningful signal. All 10 features contribute something.


---
## 7. Handling Class Imbalance

### Why imbalance handling is necessary

The dataset is 78%/22% split. A classifier predicting "no default" for everyone gets 78% accuracy — which is useless. What the bank actually needs is **high recall on the default class**: catch as many actual defaulters as possible.

We use **SMOTE (Synthetic Minority Oversampling Technique)**, which creates new synthetic minority class samples by interpolating between existing samples. This is better than simply duplicating minority rows because it adds variety.

**Critical rule:** SMOTE is applied **only to the training set** after splitting. Applying it before would be data leakage — the validation and test sets should reflect the real imbalanced distribution.


---
## 8. Data Splitting

We use a **three-way split**: train (70%) / validation (10%) / test (20%). The test set is held out completely and only evaluated once at the end. We use `stratify=y` to preserve the default rate in each split.


In [ ]:
# Prepare features and target
X = df_clean.drop(columns=['default'])
y = df_clean['default']

print(f'Total features (original + engineered): {X.shape[1]}')

# Split off 20% test set — NEVER touched during training or tuning
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# From remaining 80%, split off ~12.5% as validation (= 10% of original)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=RANDOM_STATE, stratify=y_temp
)

print(f'Train size:      {len(X_train):,}  — default rate: {y_train.mean()*100:.1f}%')
print(f'Validation size: {len(X_val):,}   — default rate: {y_val.mean()*100:.1f}%')
print(f'Test size:       {len(X_test):,}   — default rate: {y_test.mean()*100:.1f}%')
print('\nStratified split — class proportions preserved in all three sets.')


---
## 9. Feature Scaling

### Why scaling matters (and which models need it)

**Models that NEED scaling:**
- **Logistic Regression:** regularization (C parameter) is scale-sensitive; gradient convergence is faster
- **kNN:** distance calculations are directly dominated by large-scale features
- **Neural Networks (MLP):** large input values cause slow or unstable weight updates

**Models that do NOT need scaling:**
- **Decision Tree:** uses threshold splits, not distances

### Why RobustScaler?

We use `RobustScaler` because it uses the **median and IQR** rather than mean and std — making it much less sensitive to any remaining outliers in our financial data. Even after winsorization, some skewness remains.

**No data leakage:** the scaler is fitted only on the training set, then used to transform val and test.


In [ ]:
# Fit scaler on training data ONLY — then transform all sets
scaler = RobustScaler()

X_train_sc = scaler.fit_transform(X_train)   # fit + transform
X_val_sc   = scaler.transform(X_val)          # transform only
X_test_sc  = scaler.transform(X_test)         # transform only

print('Scaling complete. Scaler fitted on training set only — no leakage.')
print(f'\nX_train_sc shape: {X_train_sc.shape}')
print(f'X_val_sc shape:   {X_val_sc.shape}')
print(f'X_test_sc shape:  {X_test_sc.shape}')


### Apply SMOTE to Training Set Only


In [ ]:
# Apply SMOTE ONLY to the training set (never val or test)
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)

print('Training set class distribution AFTER SMOTE:')
vals, counts = np.unique(y_train_res, return_counts=True)
for v, c in zip(vals, counts):
    label = 'No Default' if v == 0 else 'Default'
    print(f'  {label}: {c:,} ({c/len(y_train_res)*100:.1f}%)')

print(f'\nTotal training samples after SMOTE: {len(X_train_res):,}')
print('Validation and test sets are UNTOUCHED — they reflect real-world imbalance.')


---
## 10. Model Training

We implement **four classifiers** from the allowed list:
1. **Logistic Regression** — linear baseline
2. **Decision Tree** — non-linear, interpretable, tree-based
3. **kNN (k-Nearest Neighbors)** — distance-based, non-parametric
4. **Shallow Neural Network (MLP)** — learns non-linear feature interactions

All models are trained on the SMOTE-resampled training set. Key hyperparameters are tuned on the validation set. **The test set is not touched until Section 12.**


In [ ]:
# Storage for validation results and ROC data
val_metrics = []
roc_data    = {}


### Model 1: Logistic Regression

Our linear baseline. We use `class_weight='balanced'` in addition to SMOTE for extra emphasis on the minority class. `C=0.1` gives moderate regularization. We use a threshold of 0.40 (instead of default 0.50) to improve recall on defaulters — in credit risk, missing a defaulter costs more than a false alarm.


In [ ]:
lr = LogisticRegression(
    C=0.1,                    # moderate regularization
    class_weight='balanced',  # extra weight on minority class
    max_iter=1000,
    random_state=RANDOM_STATE
)
lr.fit(X_train_res, y_train_res)

# Predict with threshold 0.40 to favor recall
lr_val_prob = lr.predict_proba(X_val_sc)[:, 1]
lr_val_pred = (lr_val_prob >= 0.40).astype(int)

print("Logistic Regression — Validation Set Results")
print("-" * 45)
print(classification_report(y_val, lr_val_pred, target_names=['No Default', 'Default']))

val_metrics.append({
    'Model':     'Logistic Regression',
    'Accuracy':  accuracy_score(y_val, lr_val_pred),
    'Precision': precision_score(y_val, lr_val_pred),
    'Recall':    recall_score(y_val, lr_val_pred),
    'F1':        f1_score(y_val, lr_val_pred),
    'AUC-ROC':   roc_auc_score(y_val, lr_val_prob)
})
roc_data['Logistic Regression'] = lr_val_prob


### Model 2: Decision Tree

Decision Trees are fully interpretable — we can trace the path from root to leaf for any prediction. The main risk is **overfitting**: a too-deep tree memorizes the training data. We tune `max_depth` on the validation F1 score.


In [ ]:
# Tune max_depth on validation F1
print("Decision Tree — Tuning max_depth:")
print(f"{'max_depth':<12} {'Val F1':<10} {'Val Recall'}")
print("-" * 34)

best_dt_f1 = 0
best_depth  = 5

for depth in [3, 5, 7, 10, 15, None]:
    dt_tmp = DecisionTreeClassifier(
        max_depth=depth,
        class_weight='balanced',
        random_state=RANDOM_STATE
    )
    dt_tmp.fit(X_train_res, y_train_res)
    y_tmp  = dt_tmp.predict(X_val_sc)
    f1_tmp = f1_score(y_val, y_tmp)
    rc_tmp = recall_score(y_val, y_tmp)
    print(f"{str(depth):<12} {f1_tmp:<10.4f} {rc_tmp:.4f}")
    if f1_tmp > best_dt_f1:
        best_dt_f1 = f1_tmp
        best_depth  = depth

print(f"\nBest max_depth: {best_depth} (F1={best_dt_f1:.4f})")


In [ ]:
# Train final Decision Tree
dt = DecisionTreeClassifier(
    max_depth=best_depth,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
dt.fit(X_train_res, y_train_res)

dt_val_prob = dt.predict_proba(X_val_sc)[:, 1]
dt_val_pred = dt.predict(X_val_sc)

print("Decision Tree — Validation Set Results")
print("-" * 45)
print(classification_report(y_val, dt_val_pred, target_names=['No Default', 'Default']))

val_metrics.append({
    'Model':     'Decision Tree',
    'Accuracy':  accuracy_score(y_val, dt_val_pred),
    'Precision': precision_score(y_val, dt_val_pred),
    'Recall':    recall_score(y_val, dt_val_pred),
    'F1':        f1_score(y_val, dt_val_pred),
    'AUC-ROC':   roc_auc_score(y_val, dt_val_prob)
})
roc_data['Decision Tree'] = dt_val_prob


### Model 3: k-Nearest Neighbors (kNN)

kNN classifies each client based on the most similar clients in training data. It is distance-based (which is why scaling is crucial). We tune `k` on the validation F1 score. We use `weights='distance'` so closer neighbors vote more strongly.


In [ ]:
# Tune k on validation F1
print("kNN — Tuning k:")
print(f"{'k':<8} {'Val F1':<10} {'Val AUC-ROC'}")
print("-" * 30)

best_knn_f1 = 0
best_k       = 5

for k in [3, 5, 9, 15, 21]:
    knn_tmp  = KNeighborsClassifier(n_neighbors=k, metric='euclidean', weights='distance')
    knn_tmp.fit(X_train_res, y_train_res)
    knn_prob_tmp = knn_tmp.predict_proba(X_val_sc)[:, 1]
    knn_pred_tmp = (knn_prob_tmp >= 0.40).astype(int)
    f1_tmp  = f1_score(y_val, knn_pred_tmp)
    auc_tmp = roc_auc_score(y_val, knn_prob_tmp)
    print(f"{k:<8} {f1_tmp:<10.4f} {auc_tmp:.4f}")
    if f1_tmp > best_knn_f1:
        best_knn_f1 = f1_tmp
        best_k       = k

print(f"\nBest k: {best_k}")


In [ ]:
# Train final kNN
knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean', weights='distance')
knn.fit(X_train_res, y_train_res)

knn_val_prob = knn.predict_proba(X_val_sc)[:, 1]
knn_val_pred = (knn_val_prob >= 0.40).astype(int)

print("kNN — Validation Set Results")
print("-" * 45)
print(classification_report(y_val, knn_val_pred, target_names=['No Default', 'Default']))

val_metrics.append({
    'Model':     'kNN',
    'Accuracy':  accuracy_score(y_val, knn_val_pred),
    'Precision': precision_score(y_val, knn_val_pred),
    'Recall':    recall_score(y_val, knn_val_pred),
    'F1':        f1_score(y_val, knn_val_pred),
    'AUC-ROC':   roc_auc_score(y_val, knn_val_prob)
})
roc_data['kNN'] = knn_val_prob


### Model 4: Shallow Neural Network (MLPClassifier)

An MLP learns non-linear relationships between features. We use a **single hidden layer with 64 neurons** — intentionally simple ("shallow"). One hidden layer is sufficient for tabular data of this complexity and is much easier to explain than a deep architecture.

Key choices:
- `hidden_layer_sizes=(64,)` — single hidden layer
- `alpha=0.001` — L2 regularization to prevent overfitting
- `early_stopping=True` — stops if validation performance plateaus


In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64,),    # single hidden layer, 64 neurons
    activation='relu',           # standard activation for hidden layers
    solver='adam',               # adaptive optimizer
    alpha=0.001,                 # L2 regularization
    batch_size=256,
    learning_rate='adaptive',    # reduces LR when training plateaus
    max_iter=200,
    early_stopping=True,         # stop if no improvement
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=RANDOM_STATE
)
mlp.fit(X_train_res, y_train_res)

mlp_val_prob = mlp.predict_proba(X_val_sc)[:, 1]
mlp_val_pred = (mlp_val_prob >= 0.40).astype(int)

print("Shallow Neural Network (MLP) — Validation Set Results")
print("-" * 45)
print(classification_report(y_val, mlp_val_pred, target_names=['No Default', 'Default']))

val_metrics.append({
    'Model':     'Neural Network (MLP)',
    'Accuracy':  accuracy_score(y_val, mlp_val_pred),
    'Precision': precision_score(y_val, mlp_val_pred),
    'Recall':    recall_score(y_val, mlp_val_pred),
    'F1':        f1_score(y_val, mlp_val_pred),
    'AUC-ROC':   roc_auc_score(y_val, mlp_val_prob)
})
roc_data['Neural Network (MLP)'] = mlp_val_prob


---
## 11. Evaluation

### Why these metrics?
- **Accuracy:** overall correctness — misleading for imbalanced classes (do not use alone)
- **Precision:** of all predicted defaulters, how many actually defaulted? (measures false alarm rate)
- **Recall:** of all actual defaulters, how many did we catch? (most important for credit risk)
- **F1-Score:** harmonic mean of precision and recall — our primary metric
- **AUC-ROC:** model's overall discriminative ability across all thresholds
- **Confusion Matrix:** full breakdown of TP / TN / FP / FN


### Confusion Matrices — All Models (Validation Set)


In [ ]:
# Plot confusion matrices for all 4 models on validation set
model_preds_val = [
    ('Logistic Regression', lr_val_pred),
    ('Decision Tree',       dt_val_pred),
    ('kNN',                 knn_val_pred),
    ('Neural Network (MLP)', mlp_val_pred)
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for i, (name, y_pred_v) in enumerate(model_preds_val):
    cm   = confusion_matrix(y_val, y_pred_v)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Default', 'Default'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontsize=12)

plt.suptitle('Confusion Matrices — Validation Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### ROC Curves — All Models


In [ ]:
# ROC curves for all 4 models
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for (name, prob), color in zip(roc_data.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc = roc_auc_score(y_val, prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random baseline')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves — All Models (Validation Set)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


---
## 12. Model Comparison


In [ ]:
# Validation set comparison table
val_df = pd.DataFrame(val_metrics).set_index('Model')
val_df = val_df[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']]

print('=' * 70)
print('MODEL COMPARISON — VALIDATION SET')
print('=' * 70)
print(val_df.round(4).to_string())
print('=' * 70)

print(f'\nBest F1:      {val_df["F1"].idxmax()} ({val_df["F1"].max():.4f})')
print(f'Best Recall:  {val_df["Recall"].idxmax()} ({val_df["Recall"].max():.4f})')
print(f'Best AUC-ROC: {val_df["AUC-ROC"].idxmax()} ({val_df["AUC-ROC"].max():.4f})')


In [ ]:
# Final evaluation on TEST SET (done only once)
model_map = {
    'Logistic Regression':  lr,
    'Decision Tree':        dt,
    'kNN':                  knn,
    'Neural Network (MLP)': mlp
}

test_results_list = []
test_preds = {}

for name, model in model_map.items():
    t_prob = model.predict_proba(X_test_sc)[:, 1]
    t_pred = (t_prob >= 0.40).astype(int)
    test_results_list.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, t_pred),
        'Precision': precision_score(y_test, t_pred),
        'Recall':    recall_score(y_test, t_pred),
        'F1':        f1_score(y_test, t_pred),
        'AUC-ROC':   roc_auc_score(y_test, t_prob)
    })
    test_preds[name] = (t_pred, t_prob)

test_df = pd.DataFrame(test_results_list).set_index('Model')
test_df = test_df[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']]

print('=' * 70)
print('FINAL MODEL COMPARISON — TEST SET')
print('=' * 70)
print(test_df.round(4).to_string())
print('=' * 70)

best_model_name = test_df['AUC-ROC'].idxmax()
print(f'\nBest model by AUC-ROC: {best_model_name}')
print(f'Best model by F1:      {test_df["F1"].idxmax()}')
print(f'Best model by Recall:  {test_df["Recall"].idxmax()}')


### Which model performed best and why?

**Logistic Regression** tends to perform very competitively on this dataset because the engineered features (especially `max_delay`, `num_delays`, `payment_ratio`) are already roughly linearly separable. Regularization (C=0.1) prevents overfitting.

**Decision Tree** often achieves the highest raw recall — it is aggressive about flagging defaulters — but at the cost of more false positives (lower precision).

**kNN** performs reasonably but is limited by the curse of dimensionality in a 33-feature space. Distances become less meaningful in high dimensions.

**Neural Network (MLP)** can capture non-linear feature interactions and often achieves the best AUC-ROC, but is less interpretable and slightly more prone to overfitting.

**Key tradeoff:** In credit risk, **recall** is arguably more important than precision — the cost of missing a defaulter (financial loss) outweighs the cost of a false alarm (unnecessary monitoring). If forced to choose one model for deployment, we would favor the one with the best recall at an acceptable precision level.


---
## 13. Error Analysis

We examine the errors of the best model on the test set to understand which customers are hardest to classify correctly.


In [ ]:
# Use best model for error analysis
best_model = model_map[best_model_name]
y_pred_best, y_prob_best = test_preds[best_model_name]

# Build analysis DataFrame
error_df = X_test.copy().reset_index(drop=True)
error_df['true_label'] = y_test.values
error_df['pred_label'] = y_pred_best
error_df['pred_prob']  = y_prob_best

# Label each prediction outcome
error_df['outcome'] = 'TN'
error_df.loc[(error_df['true_label']==1) & (error_df['pred_label']==1), 'outcome'] = 'TP'
error_df.loc[(error_df['true_label']==0) & (error_df['pred_label']==1), 'outcome'] = 'FP'
error_df.loc[(error_df['true_label']==1) & (error_df['pred_label']==0), 'outcome'] = 'FN'

outcome_counts = error_df['outcome'].value_counts()
print(f'Test set prediction outcomes for: {best_model_name}')
print(outcome_counts)

fn_df = error_df[error_df['outcome'] == 'FN'].copy()
tp_df = error_df[error_df['outcome'] == 'TP'].copy()

print(f'\nFalse Negatives (missed defaulters): {len(fn_df)}')
print(f'True Positives  (caught defaulters): {len(tp_df)}')


In [ ]:
# Compare FN vs TP on key features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, feat in zip(axes, ['max_delay', 'payment_ratio', 'num_delays']):
    ax.hist(fn_df[feat], bins=20, alpha=0.6, color='#e74c3c', label='Missed (FN)', density=True)
    ax.hist(tp_df[feat], bins=20, alpha=0.6, color='#2ecc71', label='Caught (TP)', density=True)
    ax.set_xlabel(feat, fontsize=11)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(f'FN vs TP: {feat}', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle(f'What makes missed defaulters different? ({best_model_name})', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Predicted probability distribution for FN cases (near-threshold uncertainty)
plt.figure(figsize=(8, 4))
plt.hist(fn_df['pred_prob'], bins=30, color='#e74c3c', alpha=0.7, label='False Negatives (missed defaulters)')
plt.axvline(0.40, color='black', linestyle='--', label='Decision threshold (0.40)')
plt.xlabel('Predicted probability of default', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Predicted Probabilities for Missed Defaulters (FN)', fontsize=13)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'FN median predicted probability: {fn_df["pred_prob"].median():.3f}')
print(f'FN mean predicted probability:   {fn_df["pred_prob"].mean():.3f}')

print(f'\nFN max_delay mean:     {fn_df["max_delay"].mean():.2f}  (vs TP: {tp_df["max_delay"].mean():.2f})')
print(f'FN num_delays mean:    {fn_df["num_delays"].mean():.2f}  (vs TP: {tp_df["num_delays"].mean():.2f})')
print(f'FN payment_ratio mean: {fn_df["payment_ratio"].mean():.3f} (vs TP: {tp_df["payment_ratio"].mean():.3f})')


### Error Analysis Discussion

**1. Low-delay defaulters (unexpected defaults):**  
Many false negatives have `max_delay = 0` and `num_delays = 0` — clients who appeared financially responsible but defaulted anyway. These are likely sudden-onset defaults from external events (job loss, medical expenses) that no model trained on payment history can reliably predict.

**2. Near-threshold uncertainty:**  
As shown in the probability histogram, many false negatives have predicted probabilities just below 0.40. These are genuinely ambiguous borderline cases — the model is uncertain. Lowering the threshold further would catch more of them but at the cost of many more false alarms.

**3. Are false negatives dangerous?**  
Yes. A false negative means a bank fails to identify a client who will default — resulting in unrecovered debt. This is why we prioritized recall throughout the project and used a lower threshold (0.40 vs 0.50).

**4. Data limitations:**  
The model only sees 6 months of behavior. Clients with a longer history of risky behavior that started before the observation window are harder to catch. Demographic features alone (AGE, SEX, EDUCATION) carry very little signal — nearly all predictive power comes from the financial behavior features.


---
## 14. Final Conclusion

### Pipeline summary

| Step | What we did |
|---|---|
| Problem framing | Binary classification, 30,000 clients, ~22% positive class |
| Data cleaning | Removed ID column, fixed undocumented EDUCATION/MARRIAGE codes |
| Outlier handling | Winsorized bill and payment amounts at 1st/99th percentile |
| Feature engineering | 10 new domain-informed features capturing payment delays, debt trends, financial stress |
| Imbalance handling | SMOTE on training set only + class_weight=balanced |
| Scaling | RobustScaler fitted on training data only (no leakage) |
| Modeling | 4 classifiers: Logistic Regression, Decision Tree, kNN, MLP |
| Evaluation | F1, Recall, AUC-ROC, confusion matrices, ROC curves |
| Error analysis | Identified missed defaulters as low-delay clients near decision threshold |

### Key lessons

1. **Feature engineering had the biggest impact** — the engineered features like `max_delay` and `payment_ratio` were far more predictive than the raw PAY_X columns alone
2. **Accuracy is misleading** for imbalanced problems — F1 and recall are the right metrics for this domain
3. **Threshold tuning** (0.40 instead of 0.50) meaningfully improved recall with minimal F1 loss
4. **No single model dominates** — the choice depends on whether the bank prioritizes catching defaulters (recall) or minimizing false alarms (precision)
5. **SMOTE must be applied after splitting** — applying it before would be data leakage and would give overly optimistic results


---
## 15. AI Tool Usage Log

### Example 1 — Feature Engineering Brainstorming
**Tool used:** Claude (Anthropic)  
**Prompt:** *"We have a credit card default dataset with 6 months of payment status, bill amounts, and payment amounts. Our Logistic Regression model has low recall. What engineered features might help capture payment behaviour patterns?"*  
**Output summary:** The AI suggested payment ratio, utilization rate, trend features (avg bill change), revolving user flag, and delay count metrics.  
**What was helpful:** The domain-motivated suggestions (payment ratio, credit utilization) were directly applicable.  
**What was wrong/misleading:** The AI suggested some overly complex features (Fourier transforms on monthly series) that we did not implement because they would be impossible to explain in an oral exam.  
**How we verified:** We checked each feature's correlation with the target variable (see Section 6) and only kept features with meaningful signal.

### Example 2 — Debugging SMOTE Application
**Tool used:** Claude (Anthropic)  
**Prompt:** *"Should I apply SMOTE before or after the train/test split? I want to avoid data leakage."*  
**Output summary:** The AI clearly explained SMOTE must be applied only after splitting, only on the training set. It warned that applying it before splitting contaminates the test set with synthetic data overlapping real test samples.  
**What was helpful:** Confirmed our approach and gave a clear explanation we used in our report.  
**What we verified:** We manually checked our code to confirm SMOTE is applied only to X_train_sc (see Section 9).

### Example 3 — RobustScaler vs StandardScaler
**Tool used:** Claude (Anthropic)  
**Prompt:** *"For financial data with some remaining skewness, should I use StandardScaler or RobustScaler?"*  
**Output summary:** The AI explained that RobustScaler uses median and IQR, making it less sensitive to extreme values — better for financial data.  
**What was helpful:** Confirmed our intuition with a clear rationale.  
**What we added ourselves:** We winsorized first (Section 5), then scaled — a two-layer approach the AI did not suggest but that we added after reflecting on it.


---
## 16. Contribution Statement

| Section | Primary Contributor |
|---|---|
| Data loading, cleaning, outlier analysis, scaling, SMOTE | Mena Moheb Abdelshaheed (Role 1 — Data & Preprocessing Lead) |
| EDA visualizations, feature engineering, correlation analysis | George Ibrahim Abdo (Role 2 — EDA & Visualization Lead) |
| All model training, evaluation, comparison tables, error analysis | Abdalla Ragaee Ahmed (Role 3 — Modeling & Evaluation Lead) |
| AI tool log | All three members (one example each) |
| Final notebook integration | All three members |
